# 01 · Define & Explore — de novo metalloprotein design, the cofactor, and the coordination scheme

**Standard slot:** *define & explore.* **For Project 24 this means:** understand de novo
metalloprotein / cofactor-binding design, then **construct the cofactor-site spec** (the cofactor +
its coordinating ligands + the coordination geometry they must hold) and run a mock
cofactor-site→scaffold hello-world (D0).

Run `00_setup.ipynb` first in this session.

## Why a de novo cofactor-binder is *the* metalloprotein benchmark
A cofactor-binding metalloprotein holds a **redox / O₂ cofactor** — a **heme** (Fe-protoporphyrin
IX), a **[4Fe-4S]** cluster, or a **Zn** — in a protein pocket via **coordinating residues**, so the
cofactor can do its job: electron transfer, O₂ transport, or catalysis. Two things make this *the*
metalloprotein design problem:
- **It is a long-standing grand challenge.** Building a clean coordination pocket for a metal/cofactor
  from scratch is hard. The field went from hand-built four-helix-bundle **"maquettes"** (DeGrado) to
  **ML-designed** cofactor-binders (Baker lab).
- **A cofactor-aware sequence designer now exists.** **LigandMPNN** can "see" a bound heme/metal and
  design the protein around it (vanilla ProteinMPNN is cofactor-blind) — which is exactly the methods
  claim this project tests.

The honest history: designed heme proteins began as low-spin four-helix-bundle maquettes that *bound*
heme but were far from natural cytochromes; **incorporation and the designed redox/O₂ behaviour are
separate, hard-won steps** beyond a good geometry on paper.

## The cofactor-site spec — the coordination motif you must build
A **cofactor-site spec** is the minimal description of the bound cofactor plus the protein ligands
that coordinate it and the geometry they must hold. This project's **default** is a **bis-His heme**
electron-transfer site (a b-type-cytochrome motif):

| Role | Residue(s) | Job at the cofactor |
|------|-----------|---------------------|
| axial ligand 1 | His (imidazole Nε2) | coordinates the heme Fe from one face (Fe–Nε2 ~2.0–2.2 Å) |
| axial ligand 2 | His (imidazole Nε2) | coordinates the heme Fe from the opposite face (His–Fe–His ~180°) |
| (the cofactor) | heme b | the 4 porphyrin N's hold Fe in-plane; the **protein** supplies the axial ligands |

Other schemes ship in `scripts/cofactor_tools.COORDINATION_SCHEMES`: **His/Met** heme (c-type),
**proximal-His + open distal** heme (O₂-binding, myoglobin-like), **[4Fe-4S]-4Cys** (ferredoxin),
**Cys2His2** (structural Zn). You **construct** the geometry from a **verified** reference structure
and/or the literature (`data/inputs/cofactor_site_def.txt`) — it is a teaching template, **not**
fabricated data, and **there are no spectra in this project.** Place the ligands around the
**cofactor**, with the right oxidation/spin state in mind (it changes the geometry).

## Setup paths

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath("../scripts"))
sys.path.insert(0, os.path.abspath("../../../shared"))
os.makedirs("results", exist_ok=True)
print("paths ready; cwd =", os.getcwd())

## Build the cofactor-site spec (mock hello-world)
`scripts/cofactor_tools.py` exposes `build_cofactor_spec(cofactor, scheme)` → a coordination-pocket
geometry spec. The distances/angles it ships are **PLACEHOLDERS** — replace them in
`data/inputs/cofactor_site_def.txt` (and in `build_cofactor_spec`) with real, cited values built from
a verified reference structure during P1. This is the **enzyme-family template** hook (Project 18,
Kemp): where Project 18 placed a base + π-stack + H-bond donor and Project 20 placed a catalytic
Zn-His3-OH, Project 24 places a **cofactor + its coordinating ligands**.

In [ ]:
from cofactor_tools import build_cofactor_spec

spec = build_cofactor_spec("heme", scheme="bis_his_heme")
print("Cofactor    :", spec.cofactor, "| function:", spec.function)
print("Coordination:", spec.cofactor_site.coordination)
print("Provenance  :", spec.provenance)
print("\nCoordinating groups (PLACEHOLDER geometry — fill from a verified structure/literature):")
for g in spec.coordinating_groups:
    ang = f"{g.target_angle}deg" if g.target_angle is not None else "n/a"
    print(f"  {g.role:16s} {g.residue}/{g.atom:4s}  d={g.target_distance}A  angle={ang}")
print("\nCoordinating residues to FIX during sequence design:", spec.coordinating_residue_ids())

## A first mock scaffold + cofactor-aware sequence (no GPU)
`scaffold_cofactor_pocket(...)` (mock) returns placeholder backbones presenting the coordination
motif; `ligandmpnn_cofactor(...)` (mock) designs sequences with the coordinating residues **fixed**
and the cofactor passed as atom context. **Every number here is SYNTHETIC** — this only proves the
plumbing runs anywhere. Switch to the real backends (RFdiffusion2/Riff-Diff on an A100; cofactor-aware
LigandMPNN CPU-fast) in `02_generate.ipynb`.

In [ ]:
from cofactor_tools import scaffold_cofactor_pocket, ligandmpnn_cofactor, coordination_geometry

scaffolds = scaffold_cofactor_pocket(spec, n=5, tool="mock")
print(f"{len(scaffolds)} mock scaffolds; example:")
print(" ", scaffolds[0])

seqs = ligandmpnn_cofactor(scaffolds[0], spec.coordinating_residue_ids(), n=3,
                           cofactor=spec.cofactor, tool="mock")
print(f"\n{len(seqs)} mock sequences for {scaffolds[0]['design_id']} "
      f"(fixed roles: {seqs[0]['fixed_coordinating_roles']}, positions {seqs[0]['fixed_positions']})")

cg = coordination_geometry(None, spec)   # mock, SYNTHETIC
print(f"\ncoordination_geometry (mock, SYNTHETIC) = {cg} A  -> pass if < 0.5 A")
print("NOTE: these are placeholder numbers. The real campaign is in notebook 02.")

## The metrics that decide a metalloprotein design
| Metric | Cutoff (`enzyme`) | Means | Does **not** mean |
|--------|-------------------|-------|-------------------|
| scRMSD | ≤ 2.0 Å | designed-vs-predicted backbone self-consistency | cofactor binds |
| pLDDT (global) | ≥ 85 | local fold confidence | thermostability / coordination |
| pLDDT (**site**) | ≥ 90 | confidence *at the coordinating residues* | the geometry is correct |
| **coordination_geom_rmsd** (`cat_geom`) | **< 0.5 Å** | predicted coordinating atoms vs the target scheme | **incorporation or function** (the cofactor may not load) |

For this project the shared filter's `cat_geom` is the **coordination**-geometry RMSD and `plddt_cat`
is the **site** confidence. The fourth row is the point of the whole project — and the last column is
the message to never forget: **coordination geometry ≠ cofactor incorporation ≠ function.** A perfect
bis-His geometry on paper does not mean heme loads, and loading does not mean the redox/O₂ behaviour.
Only **spectroscopy** (UV-vis Soret / EPR + a cofactor titration) confirms it (notebook 05).

> Reminder: **AF2 does NOT place the metal/cofactor** — it predicts the apo backbone. You dock or
> superpose the cofactor to place the metal *before* scoring the coordination geometry.

## D0 checklist
- [ ] Half-page on de novo metalloprotein / cofactor-binding design + the honest hit-rate history.
- [ ] 1-page problem statement with **measurable** success criteria + the controls you'll need
      (apo protein; a coordinating-residue→Ala mutant).
- [ ] Cofactor-site spec started in `data/inputs/cofactor_site_def.txt` (replace the PLACEHOLDERs,
      cite the verified reference structure for every distance/angle + the oxidation/spin state).
- [ ] Reproduced mock hello-world (coordinating-group spec + a mock scaffold record).
- [ ] `LOG.md` entry (tool versions, GPU, seed).

**Next:** `02_generate.ipynb` — scaffold the pocket and run cofactor-aware LigandMPNN with the
coordinating residues fixed.